# 07f Theta, Prophet and AutoTS

Three more contenders on the 150 products, each built differently. Theta splits a series into a long-run line and
short-run curvature and averages the two forecasts; it did better than most methods in a large forecasting competition of 3,003 series, with almost no settings. Prophet fits a
trend, weekly and yearly seasonality and regressors as one curve; here it gets the SNAP, event and price regressors. AutoTS searches over many
time-series models by itself and picks one. We restrict it to time-series models only.

## Load

In [1]:
import os, sys, time, json, warnings, logging
sys.path.append("../src")
os.environ["PYTHONPATH"] = os.path.abspath("../src")    # the joblib workers need src/ too
warnings.filterwarnings("ignore")
for n in ["cmdstanpy", "prophet", "autots"]:
    logging.getLogger(n).disabled = True

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ROOT, PROCESSED, TABLES, FIGS, EVAL_ORIGINS, FIG_STYLE, SEED, FIT_DAYS
import ts_models as tm
from evaluation import mase_scales

RERUN = False
N_JOBS = 10
FITS = PROCESSED / "ts_fits"
plt.rcParams.update(FIG_STYLE)
written = []

inp = tm.load_inputs()
series, flags, prices, sample = inp["series"], inp["flags"], inp["prices"], inp["sample"]
scales = mase_scales(inp["daily"], tm.ALL_ORIGINS)

def scores(fc):
    val = tm.per_item(tm.item_scores(tm.for_tuning(fc), inp["actuals"], scales), tm.VAL_ORIGINS)
    sco = tm.per_item(tm.item_scores(tm.for_scoring(fc), inp["actuals"], scales), EVAL_ORIGINS)
    return tm.medians(val).iloc[0], tm.medians(sco).iloc[0]

rows, fcs = [], {}

def step(family, label, name, spec):
    t0 = time.time()
    fc = tm.run_model(name, spec, series, tm.ALL_ORIGINS, FITS, flags, prices, N_JOBS, RERUN)
    fcs[name] = fc
    v, s = scores(fc)
    st = tm.fit_stats(fc).iloc[0]
    rows.append({"family": family, "step": label, "model": name, "kept": False, **{f"val_{k}": v[k] for k in v.index},
                 **{f"scored_{k}": s[k] for k in s.index}, "fallback_rate": st["fallback_rate"]})
    print(f"{label:<28} val RMSE {v['rmse']:.3f} MAE {v['mae']:.3f} | scored RMSE {s['rmse']:.3f} MAE {s['mae']:.3f} "
          f"| fell back {st['fallback_rate']:.1%} ({(time.time() - t0) / 60:.1f} min)")
    return v

def helps(new, old):
    return new["rmse"] < old["rmse"] and new["mae"] < old["mae"]

def not_worse(new, old):
    return new["rmse"] <= old["rmse"] and new["mae"] <= old["mae"]

## Theta

`ThetaModel` with period 7. It first tests whether the series is seasonal and, if so, divides out a classical weekly
decomposition (multiplicative, unless the series has zero days, then additive), forecasts the adjusted series and puts
the season back. The only tweak the plan allows is clipping extreme training days.

In [2]:
th = {"kind": "theta"}
best = step("Theta", "Theta", "theta_base", th)
v = step("Theta", "Theta + clipping", "theta_clip", {**th, "clip": True})
keep_clip = helps(v, best)
rows[-1]["kept"] = keep_clip
theta_name = "theta_clip" if keep_clip else "theta_base"
print("Theta contender:", theta_name)

Theta                        val RMSE 4.691 MAE 3.759 | scored RMSE 4.541 MAE 3.510 | fell back 0.0% (0.0 min)


Theta + clipping             val RMSE 4.692 MAE 3.774 | scored RMSE 4.531 MAE 3.509 | fell back 0.0% (0.0 min)
Theta contender: theta_base


## Prophet

Weekly and yearly seasonality, the last 730 days, and three regressors: SNAP days, event days (other than
Christmas) and relative price. Then, one at a time: clipping, a Christmas regressor, and a yearly pattern with three
Fourier pairs instead of Prophet's default ten.

In [3]:
pro = {"cur": {"kind": "prophet", "exog": ["snap", "event", "rel_price"]}, "name": "prophet_base"}
best = step("Prophet", "Prophet with regressors", "prophet_base", pro["cur"])
cands = [("+ clipping", "clip", lambda s: {**s, "clip": True}, helps),
         ("+ Christmas regressor", "xmas", lambda s: {**s, "exog": s["exog"] + ["christmas"]}, not_worse),
         ("+ yearly order 3", "y3", lambda s: {**s, "yearly": 3}, helps)]
for label, tag, change, rule in cands:
    name = pro["name"].replace("_base", "") + "_" + tag
    spec = change(pro["cur"])
    v = step("Prophet", label, name, spec)
    if rule(v, best):
        rows[-1]["kept"] = True
        pro.update(cur=spec, name=name)
        best = v
print("Prophet contender:", pro["name"], pro["cur"])

Prophet with regressors      val RMSE 4.791 MAE 3.873 | scored RMSE 4.985 MAE 3.881 | fell back 0.0% (0.0 min)


+ clipping                   val RMSE 4.832 MAE 3.873 | scored RMSE 4.985 MAE 3.891 | fell back 0.0% (0.0 min)


+ Christmas regressor        val RMSE 4.774 MAE 3.868 | scored RMSE 4.913 MAE 3.859 | fell back 0.0% (0.0 min)


+ yearly order 3             val RMSE 4.814 MAE 3.934 | scored RMSE 4.773 MAE 3.736 | fell back 0.0% (0.0 min)
Prophet contender: prophet_xmas {'kind': 'prophet', 'exog': ['snap', 'event', 'rel_price', 'christmas']}


## AutoTS

AutoTS runs its own search: it tries many models and transformations on the last part of the training data, keeps
the best, checks them on two more backward windows, and forecasts with the winner. We give it the 150 products in
one wide table, once per origin, with data up to that origin only (the last 730 days). These are the models it may
use, all time-series models:

`ConstantNaive, LastValueNaive, AverageValueNaive, SeasonalNaive, ETS, ARIMA, Theta, UnobservedComponents,
DynamicFactor, VAR, VECM`

No regression, tree, boosting, motif or neural models. Transformations come from AutoTS's `superfast` list, one at a
time. Other settings: `forecast_length=28`, daily frequency, ensembles off, 2 generations, 2 backward validations,
seed 42. With ensembles off AutoTS picks one model for all 150 products at each origin.

The plan said: time a smoke test on 5 products and one origin first, and if the full run would take more than 8
hours, cut it down and say exactly how. The code below prints that projection. Our cut, if needed, uses AutoTS's own
`subset` option: in each validation round it scores the candidate models on a random 30 of the 150 products, and the
model it picks is then fitted and forecast on all 150. The model list, generations, validations and origins stay the
same. Because of the subset, the "best per product" table only covers the products that were in a validation subset.

In [4]:
from autots import AutoTS

# time-series models only: no regression, tree, boosting, motif or neural models
MODELS = ["ConstantNaive", "LastValueNaive", "AverageValueNaive", "SeasonalNaive", "ETS", "ARIMA", "Theta",
          "UnobservedComponents", "DynamicFactor", "VAR", "VECM"]
BASE = dict(forecast_length=28, frequency="D", ensemble=None, model_list=MODELS, transformer_list="superfast",
            transformer_max_depth=1, max_generations=2, num_validations=2, validation_method="backwards",
            random_seed=SEED, n_jobs=N_JOBS, verbose=0, no_negatives=True)
wide = inp["long_df"][inp["long_df"]["item_id"].isin(sample["item_id"])].pivot(index="date", columns="item_id", values="sales")
AT = FITS / "autots"
AT.mkdir(parents=True, exist_ok=True)


def autots_run(o, ids, settings):
    """One AutoTS search and forecast at origin o, on the last 730 days up to o only."""
    tr = wide.loc[wide.index <= o, ids].tail(FIT_DAYS)
    t0 = time.time()
    m = AutoTS(**settings).fit(tr)
    p = m.predict().forecast.clip(lower=0)
    secs = time.time() - t0
    # the model that scored best for each product in AutoTS's own validation (only models that ran every round)
    r = m.results()
    done = r.groupby("ID")["ValidationRound"].nunique()
    done = done[done == done.max()].index
    ps = m.initial_results.per_series_mae
    ps = ps[ps.index.isin(done)].groupby(level=0).mean()
    names = r.drop_duplicates("ID").set_index("ID")["Model"]
    best = ps.apply(lambda c: c.idxmin() if c.notna().any() else None).map(names)    # None: not in a validation subset
    rows = pd.DataFrame({"item_id": p.columns, "origin": o, "chosen_model": m.best_model_name,
                         "best_for_product": best.reindex(p.columns).to_numpy(), "seconds": secs})
    fc = p.reset_index(names="date").melt(id_vars="date", var_name="item_id", value_name="forecast")
    fc = fc.assign(origin=o, model="autots", converged=True, fallback=False)
    return fc, rows, m


# smoke test: 5 products, the warm-up origin, the full settings
smoke_path = AT / "smoke.json"
if RERUN or not smoke_path.exists():
    _, sm_rows, _ = autots_run(tm.VAL_ORIGINS[-1], list(sample["item_id"][:5]), BASE)
    smoke_path.write_text(json.dumps({"seconds": float(sm_rows["seconds"].iloc[0]), "chosen": sm_rows["chosen_model"].iloc[0]}))
smoke = json.loads(smoke_path.read_text())
projected = smoke["seconds"] * (150 / 5) * len(tm.ALL_ORIGINS) / 3600
print(f"smoke test: {smoke['seconds'] / 60:.1f} min for 5 products at one origin (chose {smoke['chosen']})")
print(f"projected for 150 products and {len(tm.ALL_ORIGINS)} origins: {projected:.1f} hours")

# over 8 hours, so the search is cut down (see the text above): AutoTS tries its candidate models on a random 30 of
# the 150 products in each validation round, then fits the winning model on all 150
SETTINGS = BASE if projected <= 8 else {**BASE, "subset": 30}
print("settings used:", {k: v for k, v in SETTINGS.items() if k != "model_list"})

parts, choices = [], []
for o in tm.ALL_ORIGINS:
    f, c = AT / f"{o.date()}.parquet", AT / f"{o.date()}_choices.csv"
    if RERUN or not f.exists():
        fc_o, rows_o, _ = autots_run(o, list(sample["item_id"]), SETTINGS)
        fc_o.to_parquet(f, index=False)
        rows_o.to_csv(c, index=False)
    parts.append(pd.read_parquet(f))
    choices.append(pd.read_csv(c, parse_dates=["origin"]))
    print(f"{o.date()}: {choices[-1]['chosen_model'].iloc[0]} ({choices[-1]['seconds'].iloc[0] / 60:.0f} min)")
autots_fc = pd.concat(parts, ignore_index=True)
ch = pd.concat(choices, ignore_index=True)
ch.to_csv(TABLES / "ts_autots_choices.csv", index=False)
written.append(TABLES / "ts_autots_choices.csv")
v, s = scores(autots_fc)
rows.append({"family": "AutoTS", "step": "AutoTS", "model": "autots", "kept": False, **{f"val_{k}": v[k] for k in v.index},
             **{f"scored_{k}": s[k] for k in s.index}, "fallback_rate": 0.0})
print(f"\nAutoTS: val RMSE {v['rmse']:.3f} MAE {v['mae']:.3f} | scored RMSE {s['rmse']:.3f} MAE {s['mae']:.3f}")
print(f"total search time {ch.drop_duplicates('origin')['seconds'].sum() / 3600:.1f} hours")
print("\nchosen per origin:", ch.drop_duplicates("origin")["chosen_model"].value_counts().to_dict())
print("best per product in AutoTS's validation:", ch["best_for_product"].value_counts().to_dict())

smoke test: 2.4 min for 5 products at one origin (chose AverageValueNaive)
projected for 150 products and 9 origins: 10.8 hours
settings used: {'forecast_length': 28, 'frequency': 'D', 'ensemble': None, 'transformer_list': 'superfast', 'transformer_max_depth': 1, 'max_generations': 2, 'num_validations': 2, 'validation_method': 'backwards', 'random_seed': 42, 'n_jobs': 10, 'verbose': 0, 'no_negatives': True, 'subset': 30}
2015-09-13: UnobservedComponents (9 min)
2015-10-11: ETS (9 min)
2015-11-08: ARIMA (9 min)
2015-12-06: ETS (14 min)
2016-01-03: AverageValueNaive (7 min)
2016-01-31: VECM (8 min)
2016-02-28: ETS (6 min)
2016-03-27: ETS (6 min)
2016-04-24: ARIMA (11 min)



AutoTS: val RMSE 4.742 MAE 3.727 | scored RMSE 4.848 MAE 3.885
total search time 1.3 hours

chosen per origin: {'ETS': 4, 'ARIMA': 2, 'UnobservedComponents': 1, 'AverageValueNaive': 1, 'VECM': 1}
best per product in AutoTS's validation: {'ARIMA': 125, 'AverageValueNaive': 125, 'SeasonalNaive': 108, 'ETS': 87, 'VECM': 84, 'UnobservedComponents': 46, 'VAR': 31, 'Theta': 23, 'LastValueNaive': 1}


## Save

In [5]:
abl = pd.DataFrame(rows)
abl.to_csv(TABLES / "ts_other_tuning.csv", index=False)
written.append(TABLES / "ts_other_tuning.csv")
parts = [fcs[theta_name].assign(model="theta", setting=theta_name), fcs[pro["name"]].assign(model="prophet", setting=pro["name"])]
if autots_fc is not None:
    parts.append(autots_fc.assign(setting="autots"))
out = pd.concat(parts, ignore_index=True)
out = out.merge(inp["actuals"], on=["item_id", "date"]).merge(sample[["item_id", "category"]], on="item_id")
out.to_parquet(TABLES / "ts_other_forecasts.parquet", index=False)
written.append(TABLES / "ts_other_forecasts.parquet")
assert out.groupby(["model", "item_id", "origin"]).size().eq(28).all() and out["forecast"].notna().all()
abl[["family", "step", "kept", "val_rmse", "val_mae", "val_mape", "scored_rmse", "scored_mae", "scored_mape", "fallback_rate"]].round(3)

,family,step,kept,val_rmse,val_mae,val_mape,scored_rmse,scored_mae,scored_mape,fallback_rate
0,Theta,Theta,False,4.691,3.759,0.632,4.541,3.510,0.602,0.0
1,Theta,Theta + clipping,False,4.692,3.774,0.628,4.531,3.509,0.602,0.0
2,Prophet,Prophet with regressors,False,4.791,3.873,0.639,4.985,3.881,0.619,0.0
3,Prophet,+ clipping,False,4.832,3.873,0.632,4.985,3.891,0.616,0.0
4,Prophet,+ Christmas regressor,True,4.774,3.868,0.640,4.913,3.859,0.622,0.0
5,Prophet,+ yearly order 3,False,4.814,3.934,0.624,4.773,3.736,0.611,0.0
6,AutoTS,AutoTS,False,4.742,3.727,0.627,4.848,3.885,0.626,0.0


## What we found

Theta is the strongest of the three, and the closest to SARIMA. With no tuning at all it scores RMSE 4.541 and MAE
3.510 on the six scored origins, against 4.445 and 3.522 for SARIMA in 07a: a little worse on RMSE, a little better on
MAE. Clipping extreme days moved validation RMSE by less than 0.001 and raised MAE (3.759 to 3.774), so it was not kept.
On validation Theta sits at 4.691, almost level with plain SARIMA (4.697).

Prophet is the weakest fitted model here. With SNAP, event and relative price as regressors it reaches validation
RMSE 4.791 and scored RMSE 4.985, about 12% above SARIMA. Clipping made validation RMSE worse (4.832). The Christmas
regressor passed under the not-worse rule (validation 4.774, MAE 3.868) and is kept; on the scored origins it helps
too (4.913), because it puts the closure day near zero. A third yearly Fourier order made validation worse (4.814,
MAE 3.934) and was dropped, although the scored RMSE went down to 4.773. Yearly terms behave differently on the
autumn validation windows and on the winter and spring scored ones: in 07d one Fourier pair helped SARIMA on
validation and hurt on the scored dates, and here it is the other way round. We keep to the rule. Prophet's trend and yearly curve need more than
two years of data to be estimated well on one product, and the regressors only add small effects.

AutoTS was cut, as the plan required. The smoke test took 2.4 minutes for 5 products at one origin, which projects to
10.8 hours for 150 products and 9 origins. With the `subset=30` cut described above the full run took 1.3 hours.
AutoTS did not settle on one model: it picked ETS at four origins, ARIMA at two, and UnobservedComponents, a plain
average and VECM at one each. Picking a plain average for all 150 products in January 2016 shows how noisy a model
search on 30 products and two short validation windows is. The result is validation RMSE 4.742 and scored RMSE 4.848
(MAE 3.885), worse than our own damped Holt-Winters with Box-Cox (scored 4.513), which is one of the models it chose from. A search over
many models does not beat a careful choice of one, at least not with a search budget we can afford.

In AutoTS's own validation table the best single model per product is spread across nine model types, with ARIMA and
the plain average most often on top. That spread is the case for routing products to different models. A routing step by tsfresh demand profiles was
tried and removed, because every product was routed to the same model (`docs/PREREGISTRATION_TS.md`, change 5).

The contenders from this notebook are Theta (base), Prophet with the Christmas regressor, and AutoTS.

## What comes next

`07g_tsfresh_profiles` groups the products into demand profiles, and `07h_time_series_comparison` puts every model
side by side.

In [6]:
# files this notebook wrote
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\ts_autots_choices.csv 77 KB
outputs\tables\ts_other_tuning.csv 2 KB
outputs\tables\ts_other_forecasts.parquet 1,169 KB
